# 7. Keyword and Dense Retrieval with LangChain

**RAG Pipeline Series — Notebook 7**

Notebooks 3 and 4 built keyword search (TF-IDF, BM25) and dense vector search from scratch — a manual `tokenized_corpus`/`BM25Okapi` loop, then a manual NumPy cosine-similarity loop. Notebook 5 replaced the dense side with a real vector store (Chroma). This notebook closes the gap on the keyword side: LangChain ships a `BM25Retriever` that wraps the exact same `rank_bm25` library notebook 3 used directly, behind the same `Retriever` interface as the Chroma-backed retriever from notebook 5.

The payoff is that **both retrieval strategies now share one interface** — `retriever.invoke(query) -> List[Document]` — which is what makes it possible to combine them (hybrid search, notebook 8) or swap one for the other without touching the rest of a RAG pipeline.

In this notebook we will:
1. Rebuild the chapter-tagged chunks from `rag_utils` (same pipeline as notebooks 3-6).
2. Wrap `rank_bm25` in LangChain's `BM25Retriever` — sparse, keyword-based retrieval.
3. Wrap the notebook 5 Chroma store as a retriever — dense, semantic retrieval.
4. Run the same keyword-style and paraphrased queries from notebooks 3-6 through both, side by side.
5. Compare where the correct chunk ranks under each retriever.

## Setup

In [ ]:
%pip install -q -U langchain langchain-community langchain-core rank_bm25 sentence-transformers langchain-huggingface langchain-chroma chromadb pandas

## 1. Recap: chapter-tagged chunks via `rag_utils`

Notebooks 1-6 each rebuilt the same loading + chapter-aware chunking pipeline by hand. `rag_utils.load_chapter_chunks()` (introduced alongside this notebook) bundles that entire pipeline — resolve the PDF path, load pages, strip headers/footers, split into chapters, chunk within each chapter — into one call.

In [ ]:
from rag_utils import maybe_colab_upload

# Only runs inside Colab. Opens a file picker; select rag.pdf.
# Safe to skip this cell if you're running locally and already have the file on disk.
maybe_colab_upload()

In [1]:
from rag_utils import load_chapter_chunks

pages, full_text, chapters, chunks = load_chapter_chunks()

print(f"{len(chapters)} chapters -> {len(chunks)} chapter-tagged chunks")
print(chunks[0].metadata)

d:\youtube\TheAIGuy\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


15 chapters -> 181 chapter-tagged chunks
{'chapter_num': '01', 'chapter_title': 'Introduction to RAG'}


## 2. Keyword retrieval: `BM25Retriever`

`langchain_community.retrievers.BM25Retriever` builds the exact same BM25 index notebook 3 built manually with `rank_bm25.BM25Okapi` — but `.from_documents()` takes LangChain `Document` objects directly (no manual tokenizing step) and exposes the standard `.invoke(query) -> List[Document]` retriever interface instead of a bespoke `bm25_search()` helper.

In [4]:
from langchain_community.retrievers import BM25Retriever

keyword_retriever = BM25Retriever.from_documents(chunks)
keyword_retriever.k = 5  # how many documents .invoke() returns

print(type(keyword_retriever).__name__, "ready -", len(chunks), "documents indexed")

BM25Retriever ready - 181 documents indexed


## 3. Dense retrieval: a Chroma-backed retriever

Same embedding model and Chroma setup as notebooks 5 and 6, via `rag_utils.get_embedder()` / `build_chroma_store()`. `.as_retriever()` (introduced in notebook 5) gives us the same `.invoke()` interface as `BM25Retriever` above — the two retrievers are now interchangeable from the caller's point of view, even though one does sparse lexical matching and the other does dense cosine similarity.

In [2]:
from rag_utils import build_chroma_store, get_embedder

embeddings = get_embedder()
vectorstore = build_chroma_store(chunks, embeddings=embeddings, collection_name="rag_pdf_chapters")
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print(type(dense_retriever).__name__, "ready -", vectorstore._collection.count(), "documents indexed")

Loading weights: 100%|██████████| 55/55 [00:01<00:00, 47.36it/s] 


VectorStoreRetriever ready - 181 documents indexed


## 4. One interface, two strategies

Both retrievers now expose `.invoke(query) -> List[Document]`. We reuse the same two target chunks and queries from notebooks 3-6, so the comparison is apples-to-apples:

- **Keyword-style query** — reuses the BM25 chunk's own vocabulary almost verbatim (Chapter 2's "Okapi Best Match 25" passage).
- **Paraphrased query** — describes Chapter 1's "dynamic, external knowledge source" hallucination-fix idea in different words.

In [5]:
bm25_chunk_idx = next(i for i, d in enumerate(chunks) if "Okapi Best Match 25" in d.page_content)
hallucination_chunk_idx = next(i for i, d in enumerate(chunks) if "dynamic, external knowledge source" in d.page_content)

keyword_query = "Okapi Best Match 25 term frequency saturation document length normalization"
paraphrase_query = "How can giving a language model outside documents stop it from making things up?"

def show(retriever, query, label):
    print(f"{label} -- query: {query!r}")
    for doc in retriever.invoke(query):
        print(f"  chapter={doc.metadata['chapter_num']} ({doc.metadata['chapter_title']})")
    print()

show(keyword_retriever, keyword_query, "BM25Retriever")
show(dense_retriever, keyword_query, "Chroma retriever")

BM25Retriever -- query: 'Okapi Best Match 25 term frequency saturation document length normalization'
  chapter=02 (Evolution of Retrieval)
  chapter=02 (Evolution of Retrieval)
  chapter=06 (Retrieval Techniques)
  chapter=02 (Evolution of Retrieval)
  chapter=05 (Vector Databases & Indexing)

Chroma retriever -- query: 'Okapi Best Match 25 term frequency saturation document length normalization'
  chapter=02 (Evolution of Retrieval)
  chapter=05 (Vector Databases & Indexing)
  chapter=08 (Re-ranking)
  chapter=06 (Retrieval Techniques)
  chapter=04 (Embeddings)



In [6]:
show(keyword_retriever, paraphrase_query, "BM25Retriever")
show(dense_retriever, paraphrase_query, "Chroma retriever")

print("Same story as notebooks 3/4: BM25 is strong on the keyword-style query and weaker on the paraphrase;")
print("the dense retriever holds up on both, because it matches meaning, not exact wording.")

BM25Retriever -- query: 'How can giving a language model outside documents stop it from making things up?'
  chapter=10 (Generation)
  chapter=01 (Introduction to RAG)
  chapter=01 (Introduction to RAG)
  chapter=01 (Introduction to RAG)
  chapter=01 (Introduction to RAG)

Chroma retriever -- query: 'How can giving a language model outside documents stop it from making things up?'
  chapter=03 (Data Ingestion)
  chapter=01 (Introduction to RAG)
  chapter=09 (Augmentation)
  chapter=03 (Data Ingestion)
  chapter=01 (Introduction to RAG)

Same story as notebooks 3/4: BM25 is strong on the keyword-style query and weaker on the paraphrase;
the dense retriever holds up on both, because it matches meaning, not exact wording.


## 5. Rank comparison: where does the correct chunk land?

Same rank-of-correct-chunk check as notebooks 3-6, now driven entirely through the shared `.invoke()` interface — no retriever-specific code needed to compute it.

In [7]:
import pandas as pd

keyword_retriever.k = 10
dense_retriever.search_kwargs["k"] = 10


def rank_of(target_idx, retriever, query):
    docs = retriever.invoke(query)
    for rank, doc in enumerate(docs, start=1):
        if doc.page_content == chunks[target_idx].page_content:
            return rank
    return f"> {len(docs)}"

rows = [
    {
        "query": "Keyword-style",
        "bm25_rank": rank_of(bm25_chunk_idx, keyword_retriever, keyword_query),
        "dense_rank": rank_of(bm25_chunk_idx, dense_retriever, keyword_query),
    },
    {
        "query": "Paraphrased",
        "bm25_rank": rank_of(hallucination_chunk_idx, keyword_retriever, paraphrase_query),
        "dense_rank": rank_of(hallucination_chunk_idx, dense_retriever, paraphrase_query),
    },
]
pd.DataFrame(rows)

,query,bm25_rank,dense_rank
0,Keyword-style,1,1
1,Paraphrased,2,> 10


## Takeaways

- LangChain's `BM25Retriever` and `vectorstore.as_retriever()` both implement the same `Retriever` interface (`.invoke(query) -> List[Document]`), even though one is sparse/lexical and the other is dense/semantic under the hood.
- That shared interface is what lets you swap retrieval strategies, or combine them, without rewriting the rest of a RAG pipeline.
- The keyword-vs-paraphrase gap from notebooks 3/4 persists exactly as before: BM25 is precise on exact wording, the dense retriever generalizes past it — neither is strictly "better," they fail on different queries.

**Next up (notebook 8):** combining both retrievers into one **hybrid search** ranking with **Reciprocal Rank Fusion (RRF)** — getting BM25's precision and dense retrieval's recall in a single result list.